# Threshold jumps + dendrogram (single patient)

Explore percolation threshold jumps and dendrograms for correlation networks.

**Legacy notebooks merged:**
- TEST_per_patient_analysis.ipynb

In [ ]:
%matplotlib inline
from lrgsglib.config.funcs import move_to_rootf
move_to_rootf(pathname="lrg_eegfc")
from lrg_eegfc.notebook import *

In [ ]:
from lrgsglib.core import compute_laplacian_properties, compute_normalized_linkage, compute_optimal_threshold, get_giant_component_leftoff
from scipy.cluster.hierarchy import dendrogram
from scipy.spatial.distance import squareform
import networkx as nx

patient = list_patients(Path('data/stereoeeg_patients'))[0]
phase = PHASE_LABELS[0]
band = BRAIN_BANDS_NAMES[2]

corr = load_corr_matrix(patient, phase, band, filter_type='abs', zero_diagonal=True)
if corr is None:
    corr_res = compute_corr_matrix(patient, phase, band, filter_type='abs', zero_diagonal=True, filter_time=5000)
    corr = corr_res.adjacency_matrix

G = nx.from_numpy_array(corr)
Th, jumps = find_threshold_jumps(G)

jump_idx = min(2, len(jumps) - 1) if len(jumps) else 0
threshold = Th[jumps[jump_idx]] if len(jumps) else np.percentile(corr, 90)

corr_tmp = corr.copy()
corr_tmp[corr_tmp < threshold] = 0
Gcc, removed = get_giant_component_leftoff(nx.from_numpy_array(corr_tmp))

spect, L, rho, Trho, tau = compute_laplacian_properties(Gcc, tau=None)
dists = squareform(Trho)
lnkgM, label_list, _ = compute_normalized_linkage(dists, Gcc, method='ward')
clTh, *_ = compute_optimal_threshold(lnkgM, scaling_factor=0.98)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

axes[0].imshow(corr_tmp, cmap='viridis')
axes[0].set_title('Thresholded correlation')

_ = dendrogram(lnkgM, ax=axes[1], color_threshold=clTh)
axes[1].axhline(clTh, color='red', linestyle='--')
axes[1].set_yscale('log')
axes[1].set_title('Dendrogram at threshold')

plt.tight_layout()
plt.show()